## Baseline YOLOv8n training (initial dataset)
# Overview
This notebook prepares Roboflow Dataset V4 and records the baseline YOLOv8n training setup for pork rasher packaging defect detection. The same dataset_preparation.py module is used by the evaluation notebook.

# Dataset preparation
*   Group augmented JPG images by their filename prefix before `.rf.`
*   Remove source overlap between train, validation and test splits, keeping test sources first.
*   Move three images from one wrinkle source into validation.
*   Check labels and image counts, then save a record of the cleaned dataset.

# Input and Output
*   Dataset: dataset/pork_rasher_v4.zip.
*   Cleaned dataset: /content/pork_rasher_dataset_clean (temporary Colab storage).
*   Preparation record: dataset/manifests/v4_baseline_preparation.json.
*   Model weights: model_runs/yolov8n_baseline/weights/best.pt (original baseline run).

# Limitations
The filename-based cleaning may miss images that share the same source but have different identifiers. Dataset V4 has no labelled wrinkle examples in its test split. The preparation record checks the cleaned dataset but does not establish how existing model weights were trained.

In [1]:
# Mount Google Drive to access dataset and training result
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys

# Base folder for the dataset, trained weights and preparation records.
PROJECT_ROOT = Path("/content/drive/MyDrive/MMA3001_v4_Project")
ZIP_PATH = PROJECT_ROOT / "dataset" / "pork_rasher_v4.zip"
PREPARED_DIR = Path("/content/pork_rasher_dataset_clean")
TRAINING_MANIFEST = PROJECT_ROOT / "dataset" / "manifests" / "v4_baseline_preparation.json"
RUNS_DIR = PROJECT_ROOT / "model_runs"
BASELINE_WEIGHTS = RUNS_DIR / "yolov8n_baseline" / "weights" / "best.pt"

# Locate the shared dataset preparation module.
module_candidates = [
    Path.cwd() / "src",
    Path("/content/src"),
    Path("/content"),
    PROJECT_ROOT / "src",
]
# Include source folders from local GitHub clones.
module_candidates.extend(sorted(Path("/content").glob("*/src")))

module_dir = next((folder for folder in module_candidates
                   if (folder / "dataset_preparation.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError(
        "dataset_preparation.py not found. Place it in "
        "MyDrive/MMA3001_v4_Project/src/ or /content/src/, "
        "or clone the GitHub repository into /content/."
    )
sys.path.insert(0, str(module_dir))
from dataset_preparation import (  # noqa: E402
    prepare_dataset, print_audit, assert_preparations_match,
)

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Original Dataset V4 ZIP missing: {ZIP_PATH}")
print("Using shared source:", module_dir / "dataset_preparation.py")
print("Dataset ZIP:", ZIP_PATH)

Using shared source: /content/drive/MyDrive/MMA3001_v4_Project/src/dataset_preparation.py
Dataset ZIP: /content/drive/MyDrive/MMA3001_v4_Project/dataset/pork_rasher_v4.zip


In [3]:
# Clean overlapping sources and reserve the wrinkle source for validation.
manifest = prepare_dataset(
    zip_path=ZIP_PATH,
    output_dir=PREPARED_DIR,
    holdout_source="frame_with_red_8644_jpg",
    expected_holdout_images=3,
    expected_dataset_version=4,
    require_labels=True,
)
clean = PREPARED_DIR
print("Cleaned data.yaml:", clean / "data.yaml")
print("Cross-split overlaps removed:", manifest["removed_overlap_images"])
print("Wrinkle-source images moved into validation:", manifest["moved_to_validation"])
print_audit(manifest["audit"])

Cleaned data.yaml: /content/pork_rasher_dataset_clean/data.yaml
Cross-split overlaps removed: {'train': 36, 'valid': 1}
Wrinkle-source images moved into validation: 3

TRAIN: 3310 images; 684 backgrounds; 0 missing label files
  loose-meat: 66 boxes
  packaging-error: 1679 boxes
  twisted-meat: 72 boxes
  unsealed: 1052 boxes
  wrinkle: 15 boxes

VALID: 122 images; 14 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 34 boxes
  twisted-meat: 3 boxes
  unsealed: 80 boxes
  wrinkle: 3 boxes

TEST: 80 images; 12 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 23 boxes
  twisted-meat: 4 boxes
  unsealed: 49 boxes
  wrinkle: 0 boxes


In [4]:
# Compare image counts with the recorded baseline split.
EXPECTED_IMAGES = {"train": 3310, "valid": 122, "test": 80}
actual_images = {split: info["images"]
                 for split, info in manifest["audit"]["splits"].items()}

assert actual_images == EXPECTED_IMAGES, (
    f"Unexpected V4 split counts: {actual_images} versus {EXPECTED_IMAGES}. "
    "Inspect ZIP version and source grouping rather than adjusting labels."
)
assert manifest["audit"]["splits"]["test"]["boxes_by_class"]["wrinkle"] == 0

print("Split counts verified.")
print("Prepared split fingerprint:", manifest["split_fingerprint_sha256"])

Split counts verified.
Prepared split fingerprint: 501d136cebd2cc8c5de2e2d4d050064f549ca88350f70a9a3ecf0cbbc38fdd69


In [5]:
import json

# Keep a reference to the cleaned dataset used by this workflow.
TRAINING_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
if TRAINING_MANIFEST.exists():
    assert_preparations_match(manifest, TRAINING_MANIFEST)
    print("Preparation record matches:", TRAINING_MANIFEST)
else:
    TRAINING_MANIFEST.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
    print("Preparation record saved:", TRAINING_MANIFEST)

Preparation record saved: /content/drive/MyDrive/MMA3001_v4_Project/dataset/manifests/v4_baseline_preparation.json


## Baseline model training
The baseline uses YOLOv8n, 30 epochs, 640-pixel images, batch size 16 and seed 42. Training is optional; the saved baseline weights can be used for evaluation without running training again.

In [6]:
# Existing weights are reused unless a new training run is enabled.
RETRAIN_BASELINE = False

if RETRAIN_BASELINE:
    try:
        from ultralytics import YOLO
    except ImportError as exc:
        raise ImportError("Ultralytics is required for training: %pip install ultralytics") from exc

    # Baseline training settings.
    model = YOLO("yolov8n.pt")
    run_name = (
        "yolov8n_baseline_reproduced"
        if (RUNS_DIR / "yolov8n_baseline").exists()
        else "yolov8n_baseline"
    )
    results = model.train(
        data=str(clean / "data.yaml"),
        epochs=30,
        imgsz=640,
        batch=16,
        device=0,
        seed=42,
        project=str(RUNS_DIR),
        name=run_name,
    )
    print("Training outputs:", results.save_dir)
else:
    print("Baseline training skipped.")
    print("Saved weights:", BASELINE_WEIGHTS)
    print("Weights found:", BASELINE_WEIGHTS.is_file())

Baseline training skipped.
Saved weights: /content/drive/MyDrive/MMA3001_v4_Project/model_runs/yolov8n_baseline/weights/best.pt
Weights found: True
